# Colab Training Notebook

Detected task: `bert_aspect_sentiment_multitask_with_rating`

Selected training design: **BERT + Aspect Sentiment Head + Rating Prediction Head**

In [ ]:
!pip install PyYAML scikit-learn seaborn transformers accelerate TorchCRF

In [ ]:
from pathlib import Path
import os
import sys

MANUAL_PROJECT_ROOT = None

def is_project_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / 'configs' / 'project_config.yaml').exists()
        and (path / 'src').exists()
    )

def candidate_roots(base: Path):
    if not base.exists() or not base.is_dir():
        return []
    roots = [base]
    level1 = [child for child in sorted(base.iterdir()) if child.is_dir()]
    roots.extend(level1)
    for child in level1:
        try:
            roots.extend(grand for grand in sorted(child.iterdir()) if grand.is_dir())
        except PermissionError:
            pass
    return roots

print('Locating uploaded project package...')
project_root = None
if MANUAL_PROJECT_ROOT:
    manual_path = Path(MANUAL_PROJECT_ROOT)
    print(f'Trying manual path: {manual_path}')
    if is_project_root(manual_path):
        project_root = manual_path
    else:
        raise FileNotFoundError(f'Manual path is not a valid project root: {manual_path}')

search_bases = [
    Path('/content'),
    Path('/content/drive/MyDrive'),
]
preferred_candidates = [
    Path('/content/colab_package'),
    Path('/content/ipen5160_meituan_nlp'),
    Path('/content/drive/MyDrive/colab_package'),
    Path('/content/drive/MyDrive/ipen5160_meituan_nlp'),
]

if project_root is None:
    print('Step 1/2: checking common upload locations...')
    for candidate in preferred_candidates:
        print(f'  Checking {candidate}')
        if is_project_root(candidate):
            project_root = candidate
            print('  Found project root via common location check.')
            break

if project_root is None:
    print('Step 2/2: scanning shallow directory levels under common bases...')
    for base in search_bases:
        print(f'  Base: {base}')
        for candidate in candidate_roots(base):
            print(f'    Trying {candidate}')
            if is_project_root(candidate):
                project_root = candidate
                print('    Found project root during shallow scan.')
                break
        if project_root is not None:
            break

if project_root is None:
    raise FileNotFoundError(
        'Could not locate the uploaded project package. '
        'Put `colab_package` under /content or /content/drive/MyDrive, '
        'or set MANUAL_PROJECT_ROOT explicitly in this cell.'
    )
os.chdir(project_root)
sys.path.insert(0, str(project_root))
print('PROJECT_ROOT =', project_root)


In [ ]:
import yaml
from pathlib import Path

config = yaml.safe_load((Path('configs') / 'project_config.yaml').read_text(encoding='utf-8'))
schema = yaml.safe_load((Path('data') / 'interim' / 'schema_summary.yaml').read_text(encoding='utf-8'))
print('Encoder:', config['model']['encoder_name'])
print('Task:', schema['recommended_task'])
print('Supports CRF:', schema['supports_token_level_sequence_labeling'])


In [ ]:
import pandas as pd
from src.token_analysis import analyze_token_lengths

token_outputs = analyze_token_lengths()
display(pd.read_csv(token_outputs['distribution']).head(20))
display(pd.read_csv(token_outputs['truncated_examples']).head(20))


In [ ]:
import yaml
from pathlib import Path

AVAILABLE_CONFIGS = {
    'maxlen_256': Path('configs/experiments/maxlen_256.yaml'),
    'maxlen_384': Path('configs/experiments/maxlen_384.yaml'),
    'maxlen_512': Path('configs/experiments/maxlen_512.yaml')
}
SELECTED_CONFIG_KEY = 'maxlen_256'
SELECTED_CONFIG_PATH = AVAILABLE_CONFIGS[SELECTED_CONFIG_KEY]
selected_config = yaml.safe_load(SELECTED_CONFIG_PATH.read_text(encoding='utf-8'))
print('Selected config:', SELECTED_CONFIG_PATH)
print('max_length =', selected_config['model']['max_length'])
print('use_dynamic_padding =', selected_config['model'].get('use_dynamic_padding', True))
print('pad_to_multiple_of =', selected_config['model'].get('pad_to_multiple_of'))
print('Training order recommendation: run maxlen_256 first; only try 384 if macro-F1 does not improve enough and >256 truncation is material.')


In [ ]:
from pathlib import Path
import shutil

for path in [
    Path('outputs/models/checkpoints'),
    Path('outputs/models/final_model'),
    Path('outputs/logs/train_log.json'),
    Path('outputs/tables/model_metrics.csv'),
    Path('outputs/tables/classification_report.csv'),
    Path('outputs/tables/rating_prediction_metrics.csv'),
    Path('outputs/tables/epoch_history.csv'),
    Path('outputs/predictions/model_predictions.csv'),
    Path('outputs/figures/training_loss_curve.png'),
    Path('outputs/figures/confusion_matrix.png'),
    Path('outputs/figures/rating_prediction_actual_vs_predicted.png'),
]:
    if path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
    elif path.exists():
        path.unlink()

print('Old checkpoints and outputs cleared.')


In [ ]:
from src.train_colab import main

main(config_path=SELECTED_CONFIG_PATH)

In [ ]:
from pathlib import Path
import shutil

exp_dir = Path('outputs/experiments') / SELECTED_CONFIG_KEY
exp_dir.mkdir(parents=True, exist_ok=True)
for sub in ['models', 'logs', 'tables', 'figures', 'predictions']:
    src = Path('outputs') / sub
    dst = exp_dir / sub
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

print('Archived outputs to', exp_dir)


In [ ]:
from pathlib import Path
for path in sorted(Path('outputs').rglob('*')):
    print(path)
